# Equivalent sites on a circle
Model of jumps between $N$ equivalent sites on a circle, for example the rotation of a methyl
group ($N = 3$) or of a ring ($N = 6$):

$ S(Q, E) = K \left[ A_0(Q)\,\delta(E) + \sum_{l=1}^{N-1} A_l(Q)\, L(E, \Gamma_l) \right] $,

where $K$ is the scale factor and $L(E, \Gamma)$ is a Lorentzian with half width at half maximum
$\Gamma$. The weights and widths are

$ A_l(Q) = \frac{1}{N} \sum_{k=1}^{N} j_0(Q r_k) \cos\left(\frac{2\pi l k}{N}\right), \quad
r_k = 2R \sin\left(\frac{\pi k}{N}\right), \quad
\Gamma_l = \frac{2\hbar}{\tau} \sin^2\left(\frac{\pi l}{N}\right) $,

with the radius $R$ of the circle, the residence time $\tau$ on a site and $j_0(x) = \sin(x)/x$.
$A_0$ is the elastic incoherent structure factor (EISF) and $A_l$ the quasi-elastic incoherent
structure factor (QISF) of order $l$. Together they add up to one. For $N = 2$ the model is the
two-site jump model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import easydynamics as edyn

%matplotlib widget

## Creating the model
With $N$ sites the model has $N - 1$ Lorentzians at each Q. The radius is fixed by default, as it
is normally known from the molecular geometry, while the residence time is free to be fitted.

In [ ]:
Q = np.linspace(0.5, 2, 7)
energy = np.linspace(-3, 3, 601)

model = edyn.EquivalentSitesCircle(
    scale=1.0,
    radius=1.0,
    residence_time=2.0,
    number_of_sites=3,
    Q=Q,
)
print(model)

## Structure factors
At low Q the EISF falls and the QISF rises, as the jumps are probed on shorter length scales. At
larger Q both oscillate, and the EISF approaches $1/N$. At every Q the two add up to one.

In [ ]:
eisf = model.calculate_EISF()
qisf = model.calculate_QISF()
print(np.allclose(eisf + qisf, 1.0))

Q_fine = np.linspace(0.01, 5, 300)
plt.figure()
plt.plot(Q_fine, model.calculate_EISF(Q_fine), label='EISF')
plt.plot(Q_fine, model.calculate_QISF(Q_fine), label='QISF')
plt.legend()
plt.xlabel('Q (1/Å)')
plt.ylabel('Structure factor')
plt.title('N = 3 sites on a circle with R = 1 Å')
plt.show()

## The number of sites
The width of order $l$ is $(2\hbar/\tau)\sin^2(\pi l/N)$. It does not depend on Q, and for $N = 2$
there is a single width of $2\hbar/\tau$. More sites give more Lorentzians, and a lower EISF at
large Q, where it approaches $1/N$.

In [ ]:
for number_of_sites in (2, 3, 6):
    other = edyn.EquivalentSitesCircle(residence_time=2.0, number_of_sites=number_of_sites)
    widths = [other.calculate_width(Q, order)[0] for order in range(1, number_of_sites)]
    print(f'N = {number_of_sites}: widths (meV) = {np.round(widths, 3)}')

## One component per Q
At each Q the model makes a ComponentCollection with a single `DeltaLorentzian` component. Its
weights follow the radius, its widths follow the residence time and its area follows the scale. They
are dependent parameters, which are not fitted themselves but are updated when the model
parameters change.

In [ ]:
component = model.get_component_collections(0)[0]
print(component.number_of_lorentzians)
print([weight.value for weight in component.weights])
print([width.value for width in component.widths])

In [ ]:
model.residence_time.value = 4.0
print([width.value for width in component.widths])
model.residence_time.value = 2.0

## The scattering function
The delta function is a spike in a single bin, so we convolve the model with a resolution to see
the full spectrum. Here the resolution is a Gaussian with a width of 0.05 meV.

In [ ]:
resolution = edyn.ComponentCollection()
resolution.append_component(edyn.Gaussian(area=1.0, width=0.05))

component_collections = model.get_component_collections()
cmap = plt.cm.jet
nQ = len(component_collections)
plt.figure()
for Q_index, collection in enumerate(component_collections):
    convolution = edyn.Convolution(
        energy=energy,
        sample_components=collection,
        resolution_components=resolution,
    )
    plt.plot(
        energy,
        convolution.convolution(),
        label=f'Q={Q[Q_index]} Å^-1',
        color=cmap(Q_index / (nQ - 1)),
    )

plt.legend()
plt.xlabel('Energy (meV)')
plt.ylabel('Intensity (arb. units)')
plt.title('Equivalent sites on a circle, convolved with the resolution')
plt.show()

## Parameters
The global parameters of the model are the scale, the radius and the residence time.

In [ ]:
model.get_global_variables()